# FilPHANGS - Figure Production

Diagnostic and publication figures for the FilPHANGS pipeline:

1. **Pipeline walkthrough** -- six panels showing each processing stage applied to one galaxy
2. **Source removal validation** -- original vs. source-removed image with circled point sources
3. **Hierarchical RGB composite** -- filament composite maps at three scales as false-colour RGB

Edit `BASE_DIR`, `GALAXY`, and `BAND` in Cell 1 to switch to a different target.


In [ ]:
# =============================================================================
# Cell 1: Configuration and shared utilities
# Edit BASE_DIR, GALAXY, and BAND here; all subsequent cells use these variables.
# =============================================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Circle, Patch
from astropy.io import fits
from pathlib import Path
from skimage.measure import regionprops, label as sk_label
from skimage.draw import disk

# -- Edit these for your environment ------------------------------------------
BASE_DIR = Path(r"C:\Users\jhoffm72\Documents\FilPHANGS\Data")
GALAXY   = "ngc0628"
BAND     = "F770W"

galaxy_dir  = BASE_DIR / f"{GALAXY}_{BAND}"
FIGURES_DIR = BASE_DIR / "Figures"
FIGURES_DIR.mkdir(exist_ok=True)

# Binary colormap: dark purple = background, yellow = filaments
BINARY_CMAP = ListedColormap([(68/255, 1/255, 84/255), (1, 1, 0)])

def find_fits(path, *fallbacks):
    for p in (Path(path), *[Path(f) for f in fallbacks]):
        if p.exists():
            return p
    tried = ", ".join(Path(p).name for p in (path, *fallbacks))
    print(f"  File not found (tried: {tried})")
    return None

def load_fits(path, hdu_idx=0):
    with fits.open(path, ignore_missing=True) as h:
        return np.nan_to_num(np.array(h[hdu_idx].data, dtype=float))

def pct_clip(img, lo=2, hi=98):
    return np.clip(img, np.percentile(img, lo), np.percentile(img, hi))

# Resolve original image: prefer starsub_starsub, fall back to starsub
orig_img_path = find_fits(
    BASE_DIR / "OriginalImages" / f"{GALAXY}_{BAND}_JWST_Emission_starsub_starsub.fits",
    BASE_DIR / "OriginalImages" / f"{GALAXY}_{BAND}_JWST_Emission_starsub.fits",
)


In [ ]:
# =============================================================================
# Cell 2: Pipeline Walkthrough Figure
# Six-panel figure showing each stage of the FilPHANGS pipeline applied to one
# galaxy at one CDD scale. Useful for methods sections in papers and talks.
# Update stage_files if your output naming differs from the FilPHANGS defaults.
# =============================================================================
SCALE   = 16         # CDD scale in parsecs
CY, CX  = 900, 700  # crop window top-left corner (row, col)
CROP_SZ = 400        # crop size in pixels

soax_dir = galaxy_dir / "SoaxOutput" / f"{SCALE}pc"

def load_soax_stack(sdir):
    """Sum all SOAX .fits outputs in sdir into a single stacked map."""
    stack = None
    for fname in sorted(os.listdir(sdir)):
        if not fname.endswith(".fits"):
            continue
        img = load_fits(sdir / fname)
        if stack is None:
            stack = img.copy()
        elif img.shape == stack.shape:
            stack += img
    return stack

# (path, title) entries -- None marks the computed stacked panel
stage_files = [
    (galaxy_dir / "CDD" /
     f"{GALAXY}_{BAND}_JWST_Emission_starsub_starsub_CDDss{SCALE:04d}pc.fits",
     f"1. {SCALE} pc CDD (arctan stretch)"),
    (galaxy_dir / "BkgSubDivRMS" /
     f"{GALAXY}_{BAND}_JWST_Emission_starsub_starsub_CDDss{SCALE:04d}pc_BkgSubDivRMS.fits",
     "2. Background-subtracted / RMS"),
    (galaxy_dir / "SoaxOutput" / f"{SCALE}pc" /
     f"{GALAXY}_{BAND}_JWST_Emission_starsub_starsub_CDDss{SCALE:04d}pc.fits_Blocked--ridge0.02375--stretch1.750.fits",
     "3. Single SOAX run"),
    (None, "4. All SOAX runs stacked"),
    (galaxy_dir / "Composites" /
     f"{GALAXY}_{BAND}_JWST_Emission_starsub_starsub_CDDss{SCALE:04d}pc_ProcessedComposite.fits",
     "5. Composite (processComposite)"),
    (galaxy_dir / "SyntheticMap" /
     f"{GALAXY}_{BAND}_JWST_Emission_starsub_starsub_CDDss{SCALE:04d}pc_SyntheticMap_Grouped.fits",
     "6. PSF synthetic map"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10), constrained_layout=True)
for i, (ax, (path, title)) in enumerate(zip(axes.flatten(), stage_files)):
    if path is None:  # panel 4: compute stacked SOAX on the fly
        try:
            stack = load_soax_stack(soax_dir)
            crop  = stack[CY:CY+CROP_SZ, CX:CX+CROP_SZ]
            ax.imshow(pct_clip(crop), cmap="viridis", origin="lower")
        except Exception as e:
            ax.text(0.5, 0.5, f"Error:\n{e}", ha="center", va="center",
                    transform=ax.transAxes, fontsize=7, color="red")
    else:
        try:
            img  = load_fits(path)
            crop = img[CY:CY+CROP_SZ, CX:CX+CROP_SZ]
            if len(np.unique(crop)) <= 2:
                ax.imshow(crop, cmap=BINARY_CMAP, origin="lower", vmin=0, vmax=1)
            else:
                ax.imshow(pct_clip(crop), cmap="viridis", origin="lower")
        except FileNotFoundError:
            ax.text(0.5, 0.5, f"File not found:\n{Path(path).name}",
                    ha="center", va="center", transform=ax.transAxes,
                    fontsize=7, color="red")
    ax.set_title(title, fontsize=11, weight="bold")
    ax.axis("off")

fig.suptitle(f"FilPHANGS Pipeline -- {GALAXY.upper()} {BAND} @ {SCALE} pc",
             fontsize=14, weight="bold")
out = FIGURES_DIR / f"Pipeline_{GALAXY}_{SCALE}pc.png"
fig.savefig(out, dpi=300)
plt.show()
print(f"Saved {out}")

In [ ]:
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from skimage.measure import regionprops, label
from matplotlib.patches import Circle
import os
from skimage.draw import disk

# -- File paths with fallbacks ------------------------------------------------
source_rem_path = find_fits(
    BASE_DIR / f"{GALAXY}_{BAND}" / "Source_Removal" / "OriginalImageSourcesRemoved.fits",
)
cell3_orig_path = find_fits(
    BASE_DIR / "OriginalImages" / f"{GALAXY}_{BAND}_JWST_Emission_starsub_starsub.fits",
    BASE_DIR / "OriginalImages" / f"{GALAXY}_{BAND}_JWST_Emission_starsub.fits",
)
mask_path = find_fits(
    BASE_DIR / f"{GALAXY}_{BAND}" / "Source_Removal" / "_CDDfs0004pix_F770W_CDDfs_sources_S2N_mask.fits",
    BASE_DIR / f"{GALAXY}_{BAND}" / "Source_Removal" / "_CDDfs0004pix_CDDfs0004pix_F770W_CDDfs_sources_S2N_mask.fits",
)

if not all([source_rem_path, cell3_orig_path, mask_path]):
    print("Source removal figure skipped: one or more required files not found.")
else:
    with fits.open(source_rem_path) as hdu:
        source_removed_image = hdu[0].data[1]

    with fits.open(cell3_orig_path) as hdu:
        orig_image = np.nan_to_num(np.array(hdu[0].data, dtype=float))

    start_x, start_y = 700, 800
    stop_x,  stop_y  = start_x + 300, start_y + 300

    mask         = fits.getdata(mask_path).astype(bool)
    labeled_mask = label(mask)
    regions      = regionprops(labeled_mask)

    crop_origin  = (start_y, start_x)
    orig_crop    = orig_image[start_y:stop_y, start_x:stop_x]
    src_rem_crop = source_removed_image[start_y:stop_y, start_x:stop_x]

    valid_regions = []
    crop_height, crop_width = orig_crop.shape
    bright_high = np.percentile(orig_crop, 95)
    bright_low  = np.percentile(orig_crop, 5)

    for region in regions:
        y, x   = region.centroid
        radius = region.equivalent_diameter / 2
        if (start_y <= y < stop_y) and (start_x <= x < stop_x):
            y_rel, x_rel = int(y - start_y), int(x - start_x)
            rr, cc = disk((y_rel, x_rel), radius, shape=(crop_height, crop_width))
            mean_orig  = np.mean(orig_crop[rr, cc])
            mean_rem   = np.mean(src_rem_crop[rr, cc])
            diff_ratio = np.abs(mean_orig - mean_rem) / (mean_orig + 1e-8)
            if mean_orig <= bright_low:
                continue
            if diff_ratio >= 0.07 or mean_orig >= bright_high:
                valid_regions.append((x, y, radius))

    print(f"{len(valid_regions)} valid circular regions.")

    bright_percentile = 97
    vmin, vmax    = np.percentile(orig_crop, [1, bright_percentile])
    bright_thresh = np.percentile(orig_crop, bright_percentile)
    max_val       = np.percentile(orig_crop, 99.8)

    def show_image_with_circles(ax, image, crop_origin, vmin, vmax, bright_thresh, max_val):
        offset_y, offset_x = crop_origin
        ax.imshow(image, cmap="gray", origin="lower", vmin=vmin, vmax=vmax)
        bright_values = np.where(image >= bright_thresh, image, 0).astype(float)
        ax.imshow(bright_values, cmap="inferno", origin="lower",
                  vmin=bright_thresh, vmax=max_val, alpha=(bright_values > 0) * 0.8)
        ax.axis("off")
        for x, y, radius in valid_regions:
            if (offset_y <= y < offset_y + image.shape[0]) and (offset_x <= x < offset_x + image.shape[1]):
                ax.add_patch(Circle((x - offset_x, y - offset_y), radius,
                                    edgecolor="red", facecolor="none", linewidth=1.5))

    fig, axs = plt.subplots(1, 2, figsize=(14, 7))
    show_image_with_circles(axs[0], orig_crop,    crop_origin, vmin, vmax, bright_thresh, max_val)
    show_image_with_circles(axs[1], src_rem_crop, crop_origin, vmin, vmax, bright_thresh, max_val)

    out_path = FIGURES_DIR / f"ValidatedCircles_{GALAXY}_{start_x}_{start_y}_Overlay.png"
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Figure saved to: {out_path}")
    plt.show()


In [ ]:
# =============================================================================
# Cell X: Multi-Scale CDD Decomposition Figure
# 6-panel grid (2 rows x 3 cols): top-left = source-removed image (the actual
# input to the CDD pipeline), remaining 5 panels = CDD at each standard scale
# (16, 32, 64, 128, 256 pc).
# All panels share the same vmin/vmax so brighter panels are genuinely brighter.
# Edit CY2/CX2/CROP_SZ2 to zoom into a different region of interest.
# =============================================================================
CDD_SCALES = [16, 32, 64, 128, 256]
CY2, CX2   = 900, 700   # crop origin (row, col)
CROP_SZ2   = 400        # crop size in pixels

src_rem_fits = find_fits(
    galaxy_dir / "Source_Removal" / "OriginalImageSourcesRemoved.fits",
    orig_img_path,
)

# -- Load all crops first so we can compute a shared intensity scale ----------
crops = {}

if src_rem_fits is not None:
    try:
        with fits.open(src_rem_fits) as h:
            raw = np.array(h[0].data, dtype=float)
        data = raw[1] if raw.ndim == 3 else raw
        crops["src"] = np.nan_to_num(data)[CY2:CY2+CROP_SZ2, CX2:CX2+CROP_SZ2]
    except Exception as e:
        print(f"  Source-removed image error: {e}")

for scale in CDD_SCALES:
    cdd_path = find_fits(
        galaxy_dir / "CDD" / f"{GALAXY}_{BAND}_JWST_Emission_starsub_starsub_CDDss{scale:04d}pc.fits",
        galaxy_dir / "CDD" / f"{GALAXY}_{BAND}_JWST_Emission_starsub_CDDss{scale:04d}pc.fits",
    )
    if cdd_path is not None:
        try:
            crops[scale] = load_fits(cdd_path)[CY2:CY2+CROP_SZ2, CX2:CX2+CROP_SZ2]
        except Exception as e:
            print(f"  CDD {scale}pc error: {e}")

# Shared scale: 2ndâ€“98th percentile of all loaded crops combined
if crops:
    all_pixels = np.concatenate([c.flatten() for c in crops.values()])
    vmin_shared = np.percentile(all_pixels, 2)
    vmax_shared = np.percentile(all_pixels, 98)
else:
    vmin_shared, vmax_shared = 0, 1

# -- Plot ---------------------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 10), constrained_layout=True)
ax_flat = axes.flatten()

# Panel 0: source-removed image
if "src" not in crops:
    ax_flat[0].text(0.5, 0.5, "Source-removed image\nnot found",
                    ha="center", va="center", transform=ax_flat[0].transAxes,
                    fontsize=8, color="red")
else:
    ax_flat[0].imshow(crops["src"], cmap="viridis", origin="lower",
                      vmin=vmin_shared, vmax=vmax_shared)
ax_flat[0].set_title("Source-removed image", fontsize=11, weight="bold")
ax_flat[0].axis("off")

# Panels 1-5: CDD at each scale
for ax, scale in zip(ax_flat[1:], CDD_SCALES):
    if scale not in crops:
        ax.text(0.5, 0.5, f"CDD {scale} pc\nnot found",
                ha="center", va="center", transform=ax.transAxes,
                fontsize=8, color="red")
    else:
        ax.imshow(crops[scale], cmap="viridis", origin="lower",
                  vmin=vmin_shared, vmax=vmax_shared)
    ax.set_title(f"CDD {scale} pc", fontsize=11, weight="bold")
    ax.axis("off")

fig.suptitle(f"Multi-Scale CDD Decomposition -- {GALAXY.upper()} {BAND}",
             fontsize=14, weight="bold")
out = FIGURES_DIR / f"CDD_Decomposition_{GALAXY}_{BAND}.png"
fig.savefig(out, dpi=300)
plt.show()
print(f"Saved {out}")


In [ ]:
# =============================================================================
# Cell 4: Multi-Scale Hierarchical RGB Composite
# Combines filament composite maps at three scales into a false-colour RGB image.
# Useful for visualising which spatial scales of structure are co-spatial.
#
# Colour key: Blue = 128 pc | Red = 64 pc | Green = 32 pc
# NOTE: uses Composite FITS files (stacked SOAX output), NOT synthetic maps.
#       For synthetic-map composites see SyntheticMap_Figure_Production.ipynb.
# =============================================================================
from scipy.ndimage import zoom

RGB_SCALES = {"blue": 128, "red": 64, "green": 32}   # scale (pc) -> channel colour

def load_composite(scale_pc):
    """Find and load the Composite FITS for this galaxy at the given scale."""
    comp_dir = galaxy_dir / "Composites"
    for fname in os.listdir(comp_dir):
        if fname.endswith(".fits") and f"{scale_pc:04d}pc" in fname:
            return np.nan_to_num(fits.getdata(comp_dir / fname).astype(float))
    raise FileNotFoundError(f"No composite for {scale_pc} pc in {comp_dir}")

def norm01(data, pct=99):
    """Normalise data to [0, 1], clipping at the given percentile."""
    lo, hi = np.min(data), np.percentile(data, pct)
    return np.clip((data - lo) / (hi - lo + 1e-9), 0, 1)

try:
    imgs   = {ch: load_composite(sc) for ch, sc in RGB_SCALES.items()}
    target = max(imgs.values(), key=lambda a: a.size).shape
    imgs   = {ch: zoom(a, (target[0]/a.shape[0], target[1]/a.shape[1]), order=1)
              for ch, a in imgs.items()}
    imgs   = {ch: norm01(a) for ch, a in imgs.items()}

    # Stack into RGB: channel order = (Red=64pc, Green=32pc, Blue=128pc)
    rgb = np.stack([imgs["red"], imgs["green"], imgs["blue"]], axis=-1)
    rgb = np.clip(rgb / (rgb.max() + 1e-9), 0, 1)

    legend = [Patch(color=ch, label=f"{sc} pc")
              for ch, sc in [("blue",128), ("red",64), ("green",32)]]

    fig, ax = plt.subplots(figsize=(9, 9))
    ax.imshow(rgb, origin="lower")
    ax.axis("off")
    ax.set_title(f"{GALAXY.upper()} {BAND} -- Multi-Scale Filament Hierarchy", fontsize=12)
    ax.legend(handles=legend, loc="lower right", framealpha=0.7, fontsize=10)

    out = FIGURES_DIR / f"{GALAXY}_{BAND}_RGB_Composite.png"
    fig.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved {out}")

except FileNotFoundError as e:
    print(f"Missing composite file: {e}")
    print("Run the FilPHANGS pipeline first to generate Composite FITS files.")


In [ ]:
# =============================================================================
# Cell: Attenuated Image Diagnostic -- Original | Hist-Eq | ProcessedComposite
# For each galaxy that has a histogram-equalized FITS (i.e. was run with an
# extinction/attenuated image), shows the three processing stages side by side.
# Runs independently after Cell 1.
# =============================================================================
from pathlib import Path

def find_histeq_files(base_dir):
    """Yield (galaxy_folder, scale, eq_path) for every hist-eq FITS on disk."""
    for gal_dir in sorted(Path(base_dir).iterdir()):
        if not gal_dir.is_dir():
            continue
        bkg_dir = gal_dir / 'BkgSubDivRMS'
        if not bkg_dir.is_dir():
            continue
        for f in sorted(bkg_dir.iterdir()):
            if f.name.endswith('_BkgSubDivRMS_Eq.fits'):
                # Extract scale from filename (e.g. CDDss0016pc)
                import re
                m = re.search(r'CDDss(\d+pc)', f.name)
                scale = m.group(1) if m else 'unknown'
                yield gal_dir.name, scale, f

saved = []
for folder, scale, eq_path in find_histeq_files(BASE_DIR):
    galaxy_dir = BASE_DIR / folder

    # Resolve original image
    orig_path = find_fits(
        BASE_DIR / 'OriginalImages' / f'{folder}_JWST_Emission_starsub_starsub.fits',
        BASE_DIR / 'OriginalImages' / f'{folder}_JWST_Emission_starsub.fits',
    )

    # ProcessedComposite for this scale
    comp_dir = galaxy_dir / 'Composites'
    comp_path = next(
        (p for p in comp_dir.iterdir()
         if scale in p.name and 'ProcessedComposite' in p.name),
        None
    ) if comp_dir.is_dir() else None

    # Load images
    panels, titles = [], []

    if orig_path:
        panels.append(load_fits(orig_path))
        titles.append('Original image')
    else:
        panels.append(None)
        titles.append('Original (not found)')

    try:
        panels.append(load_fits(eq_path))
        titles.append(f'Histogram equalized ({scale})')
    except Exception as e:
        panels.append(None)
        titles.append(f'Hist-eq error: {e}')

    if comp_path:
        try:
            panels.append(load_fits(comp_path))
            titles.append(f'ProcessedComposite ({scale})')
        except Exception as e:
            panels.append(None)
            titles.append(f'Composite error: {e}')
    else:
        panels.append(None)
        titles.append('ProcessedComposite (not found)')

    fig, axs = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
    for ax, img, title in zip(axs, panels, titles):
        if img is not None:
            ax.imshow(pct_clip(img), cmap='gray', origin='lower')
        else:
            ax.text(0.5, 0.5, title, ha='center', va='center',
                    transform=ax.transAxes, fontsize=9, color='red')
        ax.set_title(title, fontsize=10)
        ax.axis('off')

    fig.suptitle(f'{folder} -- {scale} Attenuated Image Diagnostic',
                 fontsize=13, weight='bold')
    out = FIGURES_DIR / f'AttenuatedDiag_{folder}_{scale}.png'
    fig.savefig(out, dpi=200, bbox_inches='tight')
    plt.show()
    saved.append(out)
    print(f'  Saved {out.name}')

if not saved:
    print('No histogram-equalized files found.',
          'These are produced when the pipeline processes extinction/attenuated images.')
else:
    print(f'{len(saved)} diagnostic figures saved to {FIGURES_DIR}')


In [ ]:
# =============================================================================
# Attenuation Four-Panel: Zoomed Original | Hist-Eq | Blocked PNG | Composite
# Produces one figure per composite scale (16, 32, 64, 128, 256 pc).
# ATT_CROP_FRAC controls zoom.
# ATT_OFFSET_X / ATT_OFFSET_Y shift the crop centre independently.
# Negative ATT_OFFSET_Y shifts the crop downward (origin="lower" convention).
# HistEq PNG needs flipud; Blocked PNG does not.
# Scale is shown only in the final (composite) panel title.
# =============================================================================
import matplotlib.image as mpimg

ATT_GALAXY      = "ngc2090"
ATT_BAND        = "F555W"
ATT_SCALES      = [16, 32, 64, 128, 256]
ATT_CROP_FRAC   = 0.15   # show central 15% of each axis
ATT_OFFSET_X    = 0.05   # horizontal shift as fraction of image width
ATT_OFFSET_Y    = -0.05  # vertical shift (negative = down, positive = up)

att_gal_dir     = BASE_DIR / f"{ATT_GALAXY}_{ATT_BAND}"
att_orig_path   = BASE_DIR / "OriginalImages" / f"{ATT_GALAXY}_{ATT_BAND}_HST_Extinction.fits"
att_histeq_path = FIGURES_DIR / f"HistEq_{ATT_GALAXY}_{ATT_BAND}_16pc.png"

def center_crop(arr, frac, offset_x=0.0, offset_y=0.0):
    h, w = arr.shape[:2]
    ch, cw = int(h * frac / 2), int(w * frac / 2)
    cy = h // 2 + int(h * offset_y)
    cx = w // 2 + int(w * offset_x)
    return arr[cy - ch : cy + ch, cx - cw : cx + cw]

def load_png(path):
    img = mpimg.imread(str(path)).astype(float)
    if img.ndim == 3:
        img = img[..., 0]
    return img

orig_data = load_fits(att_orig_path)
vlo, vhi  = np.percentile(orig_data[np.isfinite(orig_data)], [2, 98])
orig_norm = np.clip((orig_data - vlo) / (vhi - vlo), 0, 1)
orig_crop = center_crop(orig_norm, ATT_CROP_FRAC, ATT_OFFSET_X, ATT_OFFSET_Y)

histeq_img  = np.flipud(load_png(att_histeq_path))
histeq_crop = center_crop(histeq_img, ATT_CROP_FRAC, ATT_OFFSET_X, ATT_OFFSET_Y)

for scale in ATT_SCALES:
    att_comp_path = (
        att_gal_dir / "Composites"
        / f"{ATT_GALAXY}_{ATT_BAND}_HST_Extinction_CDDss{scale:04d}pc_ProcessedComposite.fits"
    )
    att_blocked_path = (
        att_gal_dir / "BlockedPng"
        / f"{ATT_GALAXY}_{ATT_BAND}_HST_Extinction_CDDss{scale:04d}pc_Blocked.png"
    )

    fig, axs = plt.subplots(1, 4, figsize=(24, 6), constrained_layout=True)

    axs[0].imshow(orig_crop, cmap="gray", origin="lower", vmin=0, vmax=1)
    axs[0].set_title(f"{ATT_GALAXY.upper()} {ATT_BAND} -- Zoomed In Original", fontsize=11, weight="bold")
    axs[0].axis("off")

    axs[1].imshow(histeq_crop, cmap="gray", origin="lower", vmin=0, vmax=1)
    axs[1].set_title("Histogram Equalized", fontsize=11, weight="bold")
    axs[1].axis("off")

    if att_blocked_path.exists():
        blocked_img = load_png(att_blocked_path)
        bmax = blocked_img.max()
        if bmax > 0:
            blocked_img = blocked_img / bmax
        blocked_crop = center_crop(blocked_img, ATT_CROP_FRAC, ATT_OFFSET_X, ATT_OFFSET_Y)
        axs[2].imshow(blocked_crop, cmap="gray", origin="lower", vmin=0, vmax=1)
    else:
        axs[2].text(0.5, 0.5, "Blocked PNG\nnot found",
                    ha="center", va="center", transform=axs[2].transAxes,
                    fontsize=9, color="red")
    axs[2].set_title("Blocked PNG", fontsize=11, weight="bold")
    axs[2].axis("off")

    if att_comp_path.exists():
        comp_data = load_fits(att_comp_path)
        comp_crop = center_crop((comp_data > 0).astype(int), ATT_CROP_FRAC, ATT_OFFSET_X, ATT_OFFSET_Y)
        axs[3].imshow(comp_crop, cmap=BINARY_CMAP, origin="lower", vmin=0, vmax=1)
    else:
        axs[3].text(0.5, 0.5, "Composite\nnot found",
                    ha="center", va="center", transform=axs[3].transAxes,
                    fontsize=9, color="red")
    axs[3].set_title(f"Composite Skeleton ({scale} pc)", fontsize=11, weight="bold")
    axs[3].axis("off")

    out = FIGURES_DIR / f"Attenuation_FourPanel_{ATT_GALAXY}_{ATT_BAND}_{scale}pc.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved {out.name}")
